In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

PyTorch: 2.10.0+cu128
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [2]:
!nvidia-smi

Sat Sep 12 08:50:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os

In [4]:
import pandas as pd

BASE = "/kaggle/input/datasets/gillesokhin/nutrition5k-dataset"

nutrition = pd.read_csv(f"{BASE}/dish_nutrition_values.csv")

print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein']


,dish_id,calories,mass,fat,carb,protein
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970
1,dish_1562688426,137.569992,88.0,8.256000,5.190000,10.297000
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000


In [5]:
import os

nutrition["image_path"] = nutrition["dish_id"].apply(
    lambda x: f"{BASE}/imagery/realsense_overhead/{x}/rgb.png"
)

# Chỉ giữ món thực sự có ảnh RGB
nutrition = nutrition[
    nutrition["image_path"].apply(os.path.exists)
].reset_index(drop=True)

print("Số món có ảnh:", len(nutrition))

display(
    nutrition[
        ["dish_id", "image_path", "calories"]
    ].head()
)

Số món có ảnh: 3244


,dish_id,image_path,calories
0,dish_1561662216,/kaggle/input/datasets/gillesokhin/nutrition5k...,300.794281
1,dish_1561662054,/kaggle/input/datasets/gillesokhin/nutrition5k...,419.438782
2,dish_1562008979,/kaggle/input/datasets/gillesokhin/nutrition5k...,382.936646
3,dish_1560455030,/kaggle/input/datasets/gillesokhin/nutrition5k...,20.590000
4,dish_1558372433,/kaggle/input/datasets/gillesokhin/nutrition5k...,74.360001


In [6]:
print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein', 'image_path']


,dish_id,calories,mass,fat,carb,protein,image_path
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970,/kaggle/input/datasets/gillesokhin/nutrition5k...
1,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593,/kaggle/input/datasets/gillesokhin/nutrition5k...
2,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387,/kaggle/input/datasets/gillesokhin/nutrition5k...
3,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000,/kaggle/input/datasets/gillesokhin/nutrition5k...
4,dish_1558372433,74.360001,143.0,0.286000,0.429000,20.020000,/kaggle/input/datasets/gillesokhin/nutrition5k...


In [7]:
df = nutrition[
    ["dish_id", "image_path", "calories"]
].copy()

display(df.head())
print("Tổng số mẫu:", len(df))

,dish_id,image_path,calories
0,dish_1561662216,/kaggle/input/datasets/gillesokhin/nutrition5k...,300.794281
1,dish_1561662054,/kaggle/input/datasets/gillesokhin/nutrition5k...,419.438782
2,dish_1562008979,/kaggle/input/datasets/gillesokhin/nutrition5k...,382.936646
3,dish_1560455030,/kaggle/input/datasets/gillesokhin/nutrition5k...,20.590000
4,dish_1558372433,/kaggle/input/datasets/gillesokhin/nutrition5k...,74.360001


Tổng số mẫu: 3244


In [8]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 2270
Val: 487
Test: 487


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class NutritionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        calories = torch.tensor(row["calories"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, calories

In [10]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
train_dataset = NutritionDataset(train_df, train_transform)
val_dataset = NutritionDataset(val_df, val_transform)
test_dataset = NutritionDataset(test_df, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

2270 487 487


In [12]:
images, calories = next(iter(train_loader))

print("Images:", images.shape)
print("Calories:", calories.shape)
print("Ví dụ calories:", calories[:5])

Images: torch.Size([64, 3, 224, 224])
Calories: torch.Size([64])
Ví dụ calories: tensor([ 81.6600, 214.5278, 671.0820,  72.3608, 557.8840])


In [13]:
# Lưu split cố định để mọi experiment sau dùng cùng dữ liệu
train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df.to_csv("/kaggle/working/val.csv", index=False)
test_df.to_csv("/kaggle/working/test.csv", index=False)

print("Saved fixed splits:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Saved fixed splits:
Train: 2270
Validation: 487
Test: 487


In [14]:
# Thống kê mô tả target calories
def calorie_stats(dataframe, split_name):
    return {
        "split": split_name,
        "n_samples": len(dataframe),
        "min_kcal": dataframe["calories"].min(),
        "max_kcal": dataframe["calories"].max(),
        "mean_kcal": dataframe["calories"].mean(),
        "std_kcal": dataframe["calories"].std(),
        "median_kcal": dataframe["calories"].median(),
    }

stats_df = pd.DataFrame([
    calorie_stats(train_df, "Train"),
    calorie_stats(val_df, "Validation"),
    calorie_stats(test_df, "Test"),
    calorie_stats(pd.concat([train_df, val_df, test_df]), "Total"),
])

display(stats_df.round(2))
stats_df.to_csv("/kaggle/working/dataset_calorie_statistics.csv", index=False)


,split,n_samples,min_kcal,max_kcal,mean_kcal,std_kcal,median_kcal
0,Train,2270,0.00,3943.33,253.31,221.40,207.40
1,Validation,487,2.30,1324.08,256.67,212.09,212.16
2,Test,487,1.15,1264.55,266.35,218.93,234.58
3,Total,3244,0.00,3943.33,255.77,219.64,210.32


# EXPERIMENTS

Phần chuẩn bị dữ liệu phía trên được giữ cố định.  
Thêm các experiment mới từ đây trở xuống để bảo đảm tất cả thí nghiệm dùng cùng dataset split.


In [32]:
# ============================================================
# EXP 5 MODEL PARALLEL
# STEP 2: BUILD INGREDIENT VOCABULARY FROM TRAIN ONLY
# ============================================================

# Chỉ giữ đúng 2 cột cần dùng
# Không dùng grams / calories / fat / carb / protein
# để tránh data leakage
safe_ingredients = (
    dish_ingredients[
        ["dish_id", "ingr_id"]
    ]
    .drop_duplicates()
    .copy()
)


# ------------------------------------------------------------
# TRAIN DISH IDs
# ------------------------------------------------------------
train_dish_ids = set(
    train_df["dish_id"]
)


# ------------------------------------------------------------
# INGREDIENTS CHỈ XUẤT HIỆN TRONG TRAIN
# ------------------------------------------------------------
train_ingr_ids = (
    safe_ingredients[
        safe_ingredients["dish_id"].isin(
            train_dish_ids
        )
    ]["ingr_id"]
    .dropna()
    .unique()
)


# ------------------------------------------------------------
# SPECIAL TOKENS
# ------------------------------------------------------------
PAD_IDX = 0
UNK_IDX = 1


# Ingredient thật bắt đầu từ index 2
ingredient_to_idx = {
    ingr_id: idx + 2
    for idx, ingr_id
    in enumerate(
        sorted(train_ingr_ids)
    )
}


VOCAB_SIZE = (
    len(ingredient_to_idx) + 2
)


# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------
print("=== INGREDIENT VOCABULARY ===")

print(
    "Number of train ingredients:",
    len(train_ingr_ids)
)

print(
    "VOCAB_SIZE:",
    VOCAB_SIZE
)

print(
    "PAD_IDX:",
    PAD_IDX
)

print(
    "UNK_IDX:",
    UNK_IDX
)

print("\nFirst 5 mappings:")

for ingr_id, idx in list(
    ingredient_to_idx.items()
)[:5]:
    print(
        ingr_id,
        "->",
        idx
    )

=== INGREDIENT VOCABULARY ===
Number of train ingredients: 195
VOCAB_SIZE: 197
PAD_IDX: 0
UNK_IDX: 1

First 5 mappings:
ingr_0000000001 -> 2
ingr_0000000002 -> 3
ingr_0000000003 -> 4
ingr_0000000004 -> 5
ingr_0000000005 -> 6


In [33]:
# ============================================================
# EXP 5 MODEL PARALLEL
# STEP 3: GROUP INGREDIENTS BY DISH + CHECK COVERAGE
# ============================================================

# ------------------------------------------------------------
# GROUP INGREDIENT IDs THEO TỪNG MÓN
# ------------------------------------------------------------
dish_to_ingredients = (
    safe_ingredients
    .groupby("dish_id")["ingr_id"]
    .apply(list)
    .to_dict()
)


# ------------------------------------------------------------
# CHECK COVERAGE
# ------------------------------------------------------------
def check_ingredient_coverage(df, name):

    total = len(df)

    has_ing = df["dish_id"].isin(
        dish_to_ingredients.keys()
    )

    available = has_ing.sum()
    missing = total - available

    print(f"\n=== {name} ===")
    print("Total dishes       :", total)
    print("Have ingredients   :", available)
    print("Missing ingredients:", missing)

    if missing > 0:
        print("\nMissing dish IDs:")
        print(
            df.loc[
                ~has_ing,
                "dish_id"
            ].head(10).tolist()
        )


check_ingredient_coverage(
    train_df,
    "TRAIN"
)

check_ingredient_coverage(
    val_df,
    "VALIDATION"
)

check_ingredient_coverage(
    test_df,
    "TEST"
)


# ------------------------------------------------------------
# CHECK UNKNOWN INGREDIENTS IN VAL / TEST
# ------------------------------------------------------------
def count_unknown_ingredients(df):

    total_ing = 0
    unknown_ing = 0

    for dish_id in df["dish_id"]:

        ingr_list = dish_to_ingredients.get(
            dish_id,
            []
        )

        for ingr_id in ingr_list:

            total_ing += 1

            if ingr_id not in ingredient_to_idx:
                unknown_ing += 1

    return total_ing, unknown_ing


val_total, val_unknown = count_unknown_ingredients(
    val_df
)

test_total, test_unknown = count_unknown_ingredients(
    test_df
)


print("\n=== UNKNOWN INGREDIENT CHECK ===")

print(
    "Validation:",
    val_unknown,
    "/",
    val_total
)

print(
    "Test:",
    test_unknown,
    "/",
    test_total
)


=== TRAIN ===
Total dishes       : 2270
Have ingredients   : 2270
Missing ingredients: 0

=== VALIDATION ===
Total dishes       : 487
Have ingredients   : 487
Missing ingredients: 0

=== TEST ===
Total dishes       : 487
Have ingredients   : 487
Missing ingredients: 0

=== UNKNOWN INGREDIENT CHECK ===
Validation: 6 / 3691
Test: 3 / 3659


In [34]:
# ============================================================
# EXP 5 MODEL PARALLEL
# STEP 4: MAX_INGREDIENTS + DATASET
# ============================================================

import random
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image


# ------------------------------------------------------------
# MAX_INGREDIENTS CHỈ TỪ TRAIN
# ------------------------------------------------------------

train_ing_counts = []

for dish_id in train_df["dish_id"]:

    ingr_list = dish_to_ingredients.get(
        dish_id,
        []
    )

    train_ing_counts.append(
        len(ingr_list)
    )


MAX_INGREDIENTS = max(
    train_ing_counts
)


print("=== INGREDIENT LENGTH ===")
print(
    "MAX_INGREDIENTS:",
    MAX_INGREDIENTS
)

print(
    "Mean ingredients/train dish:",
    sum(train_ing_counts) / len(train_ing_counts)
)


# ------------------------------------------------------------
# RGB TRANSFORM
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# DATASET
# ============================================================

class NutritionVisualIngredientDataset(Dataset):

    def __init__(
        self,
        dataframe,
        dish_to_ingredients,
        ingredient_to_idx,
        max_ingredients,
        transform
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.dish_to_ingredients = (
            dish_to_ingredients
        )

        self.ingredient_to_idx = (
            ingredient_to_idx
        )

        self.max_ingredients = (
            max_ingredients
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]


        # ====================================================
        # RGB
        # ====================================================

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        image = self.transform(
            image
        )


        # ====================================================
        # INGREDIENT IDs
        # ====================================================

        ingr_list = (
            self.dish_to_ingredients.get(
                row["dish_id"],
                []
            )
        )


        encoded = []

        for ingr_id in ingr_list:

            encoded.append(
                self.ingredient_to_idx.get(
                    ingr_id,
                    UNK_IDX
                )
            )


        # Cắt nếu dài hơn MAX
        encoded = encoded[
            :self.max_ingredients
        ]


        # Pad nếu ngắn hơn MAX
        if len(encoded) < self.max_ingredients:

            encoded += [
                PAD_IDX
            ] * (
                self.max_ingredients
                - len(encoded)
            )


        ingredient_ids = torch.tensor(
            encoded,
            dtype=torch.long
        )


        # ====================================================
        # TARGET
        # ====================================================

        calories = torch.tensor(
            row["calories"],
            dtype=torch.float32
        )


        return (
            image,
            ingredient_ids,
            calories
        )


# ============================================================
# CREATE DATASETS
# ============================================================

train_ing_dataset = (
    NutritionVisualIngredientDataset(
        train_df,
        dish_to_ingredients,
        ingredient_to_idx,
        MAX_INGREDIENTS,
        train_transform
    )
)

val_ing_dataset = (
    NutritionVisualIngredientDataset(
        val_df,
        dish_to_ingredients,
        ingredient_to_idx,
        MAX_INGREDIENTS,
        eval_transform
    )
)

test_ing_dataset = (
    NutritionVisualIngredientDataset(
        test_df,
        dish_to_ingredients,
        ingredient_to_idx,
        MAX_INGREDIENTS,
        eval_transform
    )
)


# ============================================================
# SANITY CHECK 1 SAMPLE
# ============================================================

image, ingr_ids, calories = (
    train_ing_dataset[0]
)

print("\n=== DATASET CHECK ===")

print(
    "Train samples:",
    len(train_ing_dataset)
)

print(
    "Val samples:",
    len(val_ing_dataset)
)

print(
    "Test samples:",
    len(test_ing_dataset)
)

print(
    "\nImage shape:",
    image.shape
)

print(
    "Ingredient shape:",
    ingr_ids.shape
)

print(
    "Ingredient dtype:",
    ingr_ids.dtype
)

print(
    "Non-PAD ingredients:",
    (ingr_ids != PAD_IDX).sum().item()
)

print(
    "Calories:",
    calories.item()
)

print(
    "Image NaN:",
    torch.isnan(image).any().item()
)

=== INGREDIENT LENGTH ===
MAX_INGREDIENTS: 34
Mean ingredients/train dish: 7.257709251101321

=== DATASET CHECK ===
Train samples: 2270
Val samples: 487
Test samples: 487

Image shape: torch.Size([3, 224, 224])
Ingredient shape: torch.Size([34])
Ingredient dtype: torch.int64
Non-PAD ingredients: 6
Calories: 351.405517578125
Image NaN: False


In [35]:
# ============================================================
# EXP 5 MODEL PARALLEL
# STEP 5: CREATE DATALOADERS + CHECK ONE BATCH
# ============================================================

from torch.utils.data import DataLoader
import torch

BATCH_SIZE = 64


train_ing_loader = DataLoader(
    train_ing_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_ing_loader = DataLoader(
    val_ing_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_ing_loader = DataLoader(
    test_ing_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# CHECK ONE BATCH
# ============================================================

images, ingredient_ids, calories = next(
    iter(train_ing_loader)
)

print("=== DATALOADER CHECK ===")

print(
    "Images shape        :",
    images.shape
)

print(
    "Ingredient IDs shape:",
    ingredient_ids.shape
)

print(
    "Calories shape      :",
    calories.shape
)

print(
    "\nImage dtype     :",
    images.dtype
)

print(
    "Ingredient dtype:",
    ingredient_ids.dtype
)

print(
    "Target dtype    :",
    calories.dtype
)

print(
    "\nImage NaN:",
    torch.isnan(images).any().item()
)

print(
    "Target NaN:",
    torch.isnan(calories).any().item()
)

print(
    "Ingredient min:",
    ingredient_ids.min().item()
)

print(
    "Ingredient max:",
    ingredient_ids.max().item()
)

print(
    "VOCAB_SIZE:",
    VOCAB_SIZE
)

print("\nNumber of batches:")

print(
    "Train:",
    len(train_ing_loader)
)

print(
    "Val  :",
    len(val_ing_loader)
)

print(
    "Test :",
    len(test_ing_loader)
)

=== DATALOADER CHECK ===
Images shape        : torch.Size([64, 3, 224, 224])
Ingredient IDs shape: torch.Size([64, 34])
Calories shape      : torch.Size([64])

Image dtype     : torch.float32
Ingredient dtype: torch.int64
Target dtype    : torch.float32

Image NaN: False
Target NaN: False
Ingredient min: 0
Ingredient max: 193
VOCAB_SIZE: 197

Number of batches:
Train: 36
Val  : 8
Test : 8


In [36]:
# ============================================================
# EXP 5 MODEL PARALLEL
# STEP 6: CREATE MODEL PARALLEL + FORWARD CHECK
#
# GPU 0:
#   RGB -> ResNet50
#
# GPU 1:
#   Ingredient IDs -> Embedding -> Mean Pooling
#   Visual Feature + Ingredient Feature
#   -> Fusion -> Calories
# ============================================================

import gc
import torch
import torch.nn as nn

from torchvision.models import (
    resnet50,
    ResNet50_Weights
)


# ============================================================
# MODEL
# ============================================================

class VisualIngredientModelParallel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        visual_device="cuda:0",
        ingredient_device="cuda:1"
    ):
        super().__init__()

        self.visual_device = torch.device(
            visual_device
        )

        self.ingredient_device = torch.device(
            ingredient_device
        )


        # ====================================================
        # GPU 0 - VISUAL BRANCH
        # ====================================================
        self.visual_encoder = resnet50(
            weights=ResNet50_Weights.DEFAULT
        )

        visual_dim = (
            self.visual_encoder.fc.in_features
        )

        self.visual_encoder.fc = nn.Identity()

        self.visual_encoder = (
            self.visual_encoder.to(
                self.visual_device
            )
        )


        # ====================================================
        # GPU 1 - INGREDIENT BRANCH
        # ====================================================
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        ).to(
            self.ingredient_device
        )


        # ====================================================
        # GPU 1 - FUSION HEAD
        # ====================================================
        self.fusion = nn.Sequential(

            nn.Linear(
                visual_dim + embedding_dim,
                512
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                512,
                1
            )

        ).to(
            self.ingredient_device
        )


    def forward(
        self,
        image,
        ingredient_ids
    ):

        # ----------------------------------------------------
        # GPU 0
        # ----------------------------------------------------
        visual_feature = (
            self.visual_encoder(
                image
            )
        )

        # [B, 2048]


        # ----------------------------------------------------
        # GPU 1
        # ----------------------------------------------------
        emb = self.embedding(
            ingredient_ids
        )

        # [B, 34, 64]


        # Padding mask
        mask = (
            ingredient_ids != PAD_IDX
        ).unsqueeze(-1)

        emb = emb * mask


        # Mean pooling
        ingredient_feature = (
            emb.sum(dim=1)
            /
            mask.sum(
                dim=1
            ).clamp(min=1)
        )

        # [B, 64]


        # ----------------------------------------------------
        # GPU 0 -> GPU 1
        # ----------------------------------------------------
        visual_feature = (
            visual_feature.to(
                self.ingredient_device,
                non_blocking=True
            )
        )


        # ----------------------------------------------------
        # FUSION ON GPU 1
        # ----------------------------------------------------
        fused = torch.cat(
            [
                visual_feature,
                ingredient_feature
            ],
            dim=1
        )

        output = self.fusion(
            fused
        )

        return output


# ============================================================
# CLEAN GPU
# ============================================================

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# CREATE MODEL
# ============================================================

model_vi_mp = VisualIngredientModelParallel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=64,
    visual_device="cuda:0",
    ingredient_device="cuda:1"
)


# ============================================================
# DEVICE CHECK
# ============================================================

print("=== MODEL DEVICE CHECK ===")

print(
    "Visual encoder:",
    next(
        model_vi_mp.visual_encoder.parameters()
    ).device
)

print(
    "Embedding:",
    next(
        model_vi_mp.embedding.parameters()
    ).device
)

print(
    "Fusion:",
    next(
        model_vi_mp.fusion.parameters()
    ).device
)


# ============================================================
# ONE BATCH
# ============================================================

images, ingredient_ids, calories = next(
    iter(train_ing_loader)
)

images = images.to(
    model_vi_mp.visual_device
)

ingredient_ids = ingredient_ids.to(
    model_vi_mp.ingredient_device
)


print("\n=== INPUT DEVICE CHECK ===")

print(
    "Images:",
    images.device
)

print(
    "Ingredients:",
    ingredient_ids.device
)


# ============================================================
# FORWARD CHECK
# ============================================================

model_vi_mp.eval()

with torch.no_grad():

    output = model_vi_mp(
        images,
        ingredient_ids
    )


print("\n=== FORWARD CHECK ===")

print(
    "Output shape:",
    output.shape
)

print(
    "Output device:",
    output.device
)

print(
    "Output NaN:",
    torch.isnan(output).any().item()
)


# ============================================================
# GPU MEMORY
# ============================================================

print("\n=== GPU MEMORY ===")

print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        2
    ),
    "GB"
)

print(
    "GPU 1 allocated:",
    round(
        torch.cuda.memory_allocated(1)
        / 1024**3,
        2
    ),
    "GB"
)

=== MODEL DEVICE CHECK ===
Visual encoder: cuda:0
Embedding: cuda:1
Fusion: cuda:1

=== INPUT DEVICE CHECK ===
Images: cuda:0
Ingredients: cuda:1

=== FORWARD CHECK ===
Output shape: torch.Size([64, 1])
Output device: cuda:1
Output NaN: False

=== GPU MEMORY ===
GPU 0 allocated: 0.7 GB
GPU 1 allocated: 0.32 GB


In [37]:
# ============================================================
# EXP 5 MODEL PARALLEL
# STEP 7: BACKWARD / GRADIENT CHECK
# ============================================================

import torch
import torch.nn as nn


# ------------------------------------------------------------
# TRAIN MODE
# ------------------------------------------------------------

model_vi_mp.train()


# ------------------------------------------------------------
# LOSS + OPTIMIZER
# foreach=False an toàn hơn với parameters nằm trên nhiều GPU
# ------------------------------------------------------------

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model_vi_mp.parameters(),
    lr=1e-4,
    foreach=False
)


# ------------------------------------------------------------
# GET ONE BATCH
# ------------------------------------------------------------

images, ingredient_ids, calories = next(
    iter(train_ing_loader)
)


# Image -> GPU 0
images = images.to(
    model_vi_mp.visual_device,
    non_blocking=True
)


# Ingredient + target -> GPU 1
ingredient_ids = ingredient_ids.to(
    model_vi_mp.ingredient_device,
    non_blocking=True
)

calories = calories.to(
    model_vi_mp.ingredient_device,
    non_blocking=True
)


# ------------------------------------------------------------
# FORWARD
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

outputs = model_vi_mp(
    images,
    ingredient_ids
).squeeze(1)


loss = criterion(
    outputs,
    calories
)


print(
    "Loss before backward:",
    loss.item()
)


# ------------------------------------------------------------
# BACKWARD
# ------------------------------------------------------------

loss.backward()


# ------------------------------------------------------------
# CHECK GRADIENTS
# ------------------------------------------------------------

visual_grad = (
    model_vi_mp
    .visual_encoder
    .conv1
    .weight
    .grad
)

embedding_grad = (
    model_vi_mp
    .embedding
    .weight
    .grad
)

fusion_grad = (
    model_vi_mp
    .fusion[0]
    .weight
    .grad
)


print("\n=== GRADIENT CHECK ===")

print(
    "Visual branch grad exists:",
    visual_grad is not None
)

print(
    "Ingredient branch grad exists:",
    embedding_grad is not None
)

print(
    "Fusion grad exists:",
    fusion_grad is not None
)


if visual_grad is not None:
    print(
        "Visual grad mean abs:",
        visual_grad.abs().mean().item()
    )

if embedding_grad is not None:
    print(
        "Ingredient grad mean abs:",
        embedding_grad.abs().mean().item()
    )

if fusion_grad is not None:
    print(
        "Fusion grad mean abs:",
        fusion_grad.abs().mean().item()
    )


# ------------------------------------------------------------
# OPTIMIZER STEP
# ------------------------------------------------------------

optimizer.step()

print(
    "\nOptimizer step: SUCCESS"
)

Loss before backward: 242.16119384765625

=== GRADIENT CHECK ===
Visual branch grad exists: True
Ingredient branch grad exists: True
Fusion grad exists: True
Visual grad mean abs: 0.0022485917434096336
Ingredient grad mean abs: 1.947204873431474e-05
Fusion grad mean abs: 0.0013255858793854713

Optimizer step: SUCCESS


In [38]:
# ============================================================
# EXP 5 MODEL PARALLEL
# STEP 8: CLEAN 1-EPOCH SMOKE TEST
# ============================================================

import gc
import time
import random
import numpy as np
import torch
import torch.nn as nn


# ============================================================
# SEED
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# CLEAN OLD MODEL
# ============================================================

if "model_vi_mp" in globals():
    del model_vi_mp

if "optimizer" in globals():
    del optimizer

gc.collect()

torch.cuda.empty_cache()


# ============================================================
# FRESH MODEL
# ============================================================

model_vi_mp = VisualIngredientModelParallel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=64,
    visual_device="cuda:0",
    ingredient_device="cuda:1"
)

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model_vi_mp.parameters(),
    lr=1e-4,
    foreach=False
)


# ============================================================
# TRAIN FUNCTION
# ============================================================

def train_one_epoch_vi_mp(
    model,
    loader,
    criterion,
    optimizer
):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for images, ingredient_ids, calories in loader:

        # RGB -> GPU 0
        images = images.to(
            model.visual_device,
            non_blocking=True
        )

        # Ingredient + target -> GPU 1
        ingredient_ids = ingredient_ids.to(
            model.ingredient_device,
            non_blocking=True
        )

        calories = calories.to(
            model.ingredient_device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        outputs = model(
            images,
            ingredient_ids
        ).squeeze(1)


        loss = criterion(
            outputs,
            calories
        )


        loss.backward()

        optimizer.step()


        batch_size = images.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size


    return (
        total_loss / total_samples
    )


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_vi_mp(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    predictions = []
    targets = []


    with torch.no_grad():

        for images, ingredient_ids, calories in loader:

            images = images.to(
                model.visual_device,
                non_blocking=True
            )

            ingredient_ids = ingredient_ids.to(
                model.ingredient_device,
                non_blocking=True
            )

            calories = calories.to(
                model.ingredient_device,
                non_blocking=True
            )


            outputs = model(
                images,
                ingredient_ids
            ).squeeze(1)


            loss = criterion(
                outputs,
                calories
            )


            batch_size = images.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            total_samples += batch_size


            predictions.append(
                outputs.detach().cpu()
            )

            targets.append(
                calories.detach().cpu()
            )


    predictions = torch.cat(
        predictions
    )

    targets = torch.cat(
        targets
    )


    mae = torch.mean(
        torch.abs(
            predictions - targets
        )
    ).item()


    rmse = torch.sqrt(
        torch.mean(
            (predictions - targets) ** 2
        )
    ).item()


    avg_loss = (
        total_loss / total_samples
    )


    return (
        avg_loss,
        mae,
        rmse
    )


# ============================================================
# RESET PEAK MEMORY
# ============================================================

torch.cuda.reset_peak_memory_stats(0)
torch.cuda.reset_peak_memory_stats(1)


# ============================================================
# TRAIN 1 EPOCH
# Training time excludes validation
# ============================================================

torch.cuda.synchronize(0)
torch.cuda.synchronize(1)

start_time = time.time()


train_loss = train_one_epoch_vi_mp(
    model_vi_mp,
    train_ing_loader,
    criterion,
    optimizer
)


torch.cuda.synchronize(0)
torch.cuda.synchronize(1)

train_time = (
    time.time() - start_time
)


# ============================================================
# TRAINING VRAM
# ============================================================

gpu0_peak = (
    torch.cuda.max_memory_allocated(0)
    / 1024**3
)

gpu1_peak = (
    torch.cuda.max_memory_allocated(1)
    / 1024**3
)


# ============================================================
# VALIDATION
# ============================================================

val_loss, val_mae, val_rmse = (
    evaluate_vi_mp(
        model_vi_mp,
        val_ing_loader,
        criterion
    )
)


# ============================================================
# THROUGHPUT
# ============================================================

throughput = (
    len(train_ing_dataset)
    / train_time
)


# ============================================================
# RESULT
# ============================================================

print(
    "=== CLEAN 1-EPOCH VISUAL-INGREDIENT "
    "MODEL PARALLEL TEST ==="
)

print(
    f"Train Loss : {train_loss:.3f}"
)

print(
    f"Val Loss   : {val_loss:.3f}"
)

print(
    f"Val MAE    : {val_mae:.2f} kcal"
)

print(
    f"Val RMSE   : {val_rmse:.2f} kcal"
)

print(
    f"Train Time : {train_time:.2f} s"
)

print(
    f"Throughput : {throughput:.2f} samples/s"
)

print(
    f"GPU 0 Peak : {gpu0_peak:.2f} GB"
)

print(
    f"GPU 1 Peak : {gpu1_peak:.2f} GB"
)

=== CLEAN 1-EPOCH VISUAL-INGREDIENT MODEL PARALLEL TEST ===
Train Loss : 241.228
Val Loss   : 126.173
Val MAE    : 126.67 kcal
Val RMSE   : 176.20 kcal
Train Time : 27.64 s
Throughput : 82.12 samples/s
GPU 0 Peak : 5.89 GB
GPU 1 Peak : 0.16 GB


In [39]:
# ============================================================
# EXP 5 MODEL PARALLEL
# STEP 9: OFFICIAL 20-EPOCH RUN
# ============================================================

import gc
import time
import random
import numpy as np
import torch
import torch.nn as nn


# ============================================================
# CONFIG
# ============================================================

SEED = 42
EPOCHS = 20
LR = 1e-4

BEST_MODEL_PATH = (
    "/kaggle/working/"
    "visual_ingredient_model_parallel_best.pth"
)


# ============================================================
# CLEAN OLD MODEL
# ============================================================

if "model_vi_mp" in globals():
    del model_vi_mp

if "optimizer" in globals():
    del optimizer

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# FRESH MODEL
# ============================================================

model_vi_mp = VisualIngredientModelParallel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=64,
    visual_device="cuda:0",
    ingredient_device="cuda:1"
)

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model_vi_mp.parameters(),
    lr=LR,
    foreach=False
)


# ============================================================
# RESULT VARIABLES
# ============================================================

best_mae = float("inf")
best_rmse = None
best_epoch = None

total_train_time = 0.0

max_gpu0_vram = 0.0
max_gpu1_vram = 0.0


# ============================================================
# TRAIN 20 EPOCHS
# ============================================================

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # Reset memory statistics
    # --------------------------------------------------------

    torch.cuda.reset_peak_memory_stats(0)
    torch.cuda.reset_peak_memory_stats(1)


    # --------------------------------------------------------
    # TRAIN
    # Validation KHÔNG nằm trong training time
    # --------------------------------------------------------

    torch.cuda.synchronize(0)
    torch.cuda.synchronize(1)

    start_time = time.time()


    train_loss = train_one_epoch_vi_mp(
        model_vi_mp,
        train_ing_loader,
        criterion,
        optimizer
    )


    torch.cuda.synchronize(0)
    torch.cuda.synchronize(1)

    epoch_train_time = (
        time.time() - start_time
    )

    total_train_time += (
        epoch_train_time
    )


    # --------------------------------------------------------
    # VRAM
    # --------------------------------------------------------

    gpu0_vram = (
        torch.cuda.max_memory_allocated(0)
        / 1024**3
    )

    gpu1_vram = (
        torch.cuda.max_memory_allocated(1)
        / 1024**3
    )


    max_gpu0_vram = max(
        max_gpu0_vram,
        gpu0_vram
    )

    max_gpu1_vram = max(
        max_gpu1_vram,
        gpu1_vram
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_loss, val_mae, val_rmse = (
        evaluate_vi_mp(
            model_vi_mp,
            val_ing_loader,
            criterion
        )
    )


    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if val_mae < best_mae:

        best_mae = val_mae
        best_rmse = val_rmse
        best_epoch = epoch + 1

        torch.save(
            model_vi_mp.state_dict(),
            BEST_MODEL_PATH
        )


    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.3f} | "
        f"Val MAE: {val_mae:.2f} | "
        f"Val RMSE: {val_rmse:.2f} | "
        f"Time: {epoch_train_time:.2f}s"
    )


# ============================================================
# OVERALL THROUGHPUT
# ============================================================

overall_throughput = (
    len(train_ing_dataset)
    * EPOCHS
    / total_train_time
)


# ============================================================
# FINAL RESULT
# ============================================================

print("\n")
print("=" * 65)

print(
    "VISUAL-INGREDIENT MODEL PARALLEL - 20 EPOCH RESULT"
)

print("=" * 65)

print(
    f"Total training time : "
    f"{total_train_time:.2f} s"
)

print(
    f"Overall throughput  : "
    f"{overall_throughput:.2f} samples/s"
)

print(
    f"Best epoch          : "
    f"{best_epoch}"
)

print(
    f"Best Val MAE        : "
    f"{best_mae:.2f} kcal"
)

print(
    f"Best Val RMSE       : "
    f"{best_rmse:.2f} kcal"
)

print(
    f"Peak GPU 0 VRAM     : "
    f"{max_gpu0_vram:.2f} GB"
)

print(
    f"Peak GPU 1 VRAM     : "
    f"{max_gpu1_vram:.2f} GB"
)

print("=" * 65)

Epoch 01/20 | Train Loss: 241.227 | Val MAE: 126.82 | Val RMSE: 175.11 | Time: 27.28s
Epoch 02/20 | Train Loss: 151.074 | Val MAE: 87.58 | Val RMSE: 119.60 | Time: 27.40s
Epoch 03/20 | Train Loss: 78.728 | Val MAE: 75.00 | Val RMSE: 113.31 | Time: 27.64s
Epoch 04/20 | Train Loss: 64.651 | Val MAE: 76.57 | Val RMSE: 114.89 | Time: 28.28s
Epoch 05/20 | Train Loss: 57.560 | Val MAE: 57.91 | Val RMSE: 91.88 | Time: 28.24s
Epoch 06/20 | Train Loss: 51.123 | Val MAE: 88.13 | Val RMSE: 133.69 | Time: 27.84s
Epoch 07/20 | Train Loss: 47.752 | Val MAE: 52.50 | Val RMSE: 83.45 | Time: 27.54s
Epoch 08/20 | Train Loss: 43.064 | Val MAE: 54.61 | Val RMSE: 87.15 | Time: 28.18s
Epoch 09/20 | Train Loss: 47.045 | Val MAE: 63.75 | Val RMSE: 102.70 | Time: 27.89s
Epoch 10/20 | Train Loss: 37.873 | Val MAE: 48.93 | Val RMSE: 79.50 | Time: 27.90s
Epoch 11/20 | Train Loss: 35.832 | Val MAE: 59.33 | Val RMSE: 88.58 | Time: 27.55s
Epoch 12/20 | Train Loss: 38.876 | Val MAE: 58.64 | Val RMSE: 89.51 | Time: 28